# PoC: Monitoramento Inteligente com VLM & Sumarização
Este notebook é o ambiente de testes para a Prova de Conceito do sistema de inteligência marítima.

## Fluxo da PoC:
1. **Detecção:** YOLOv12 identifica o navio e salva o recorte (crop).
2. **Persistência:** Registro inicial no SQLite com status `pendente`.
3. **VLM (Vision Language Model):** Analisa a imagem salva para identificar tipo, cor e detalhes.
4. **Sumarização:** Consolida as análises em um resumo narrativo do dia.

In [ ]:
import cv2
import yt_dlp
import os
import sqlite3
import pandas as pd
import ollama  # Import para usar modelos locais
from datetime import datetime
from ultralytics import YOLO
from IPython.display import display, Image
from dotenv import load_dotenv

# Carrega as variáveis de ambiente do arquivo .env
load_dotenv()

# setup config
DB_PATH = "poc_maritime.db"
CROPS_DIR = "../assets/detections"
MODEL_PATH = '../assets/yolo12m.pt'

# Modelos Ollama
VLM_MODEL = "llava"
LLM_MODEL = "llama3"

# cria pasta para as imagens
if not os.path.exists(CROPS_DIR):
    os.makedirs(CROPS_DIR)

# inicia o banco de dados SQLite
def init_db():
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS detections (
                id INTEGER PRIMARY KEY AUTOINCREMENT, 
                track_id INTEGER, 
                timestamp TEXT, 
                image_path TEXT, 
                vlm_description TEXT, 
                status TEXT DEFAULT 'pendente' 
            )
        ''')
        conn.commit()

init_db()

### Configuração da VLM (Ollama Local)
Utilizando o modelo LLaVA (Large Language-and-Vision Assistant) rodando localmente via Ollama para processar as imagens.

In [ ]:
# função de envio para a VLM usando Ollama
def analyze_image_with_vlm(image_path):
    """Envia uma foto para o Ollama (LLaVA) e recebe uma descrição textual."""
    try:
        response = ollama.chat(
            model=VLM_MODEL,
            messages=[{
                'role': 'user',
                'content': 'Descreva este navio em uma frase curta, focando no tipo (cargueiro, petroleiro, etc), cor predominante e qualquer identificação visível.',
                'images': [image_path]
            }]
        )
        return response['message']['content'].strip()
    except Exception as e:
        return f"Erro na análise VLM (Ollama): {e}"

### 1. Loop de Detecção (YOLOv12)
Célula para capturar alguns navios

tecla de parada 'q'

In [ ]:
yolo_model = YOLO(MODEL_PATH)
url = 'https://www.youtube.com/watch?v=tMYtrEBNVAU'

ydl_opts = {'format': 'best[height<=720]', 'quiet': True}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)
    video_url = info['url']

cap = cv2.VideoCapture(video_url)
seen = set()
count = 0
max_detections = 5

while count < max_detections:
    ret, frame = cap.read()
    if not ret: break
    results = yolo_model.track(frame, conf=0.6, persist=True, verbose=False)
    for result in results:
        if result.boxes.id is not None:
            for box in result.boxes:
                track_id = int(box.id.item())
                cls = int(box.cls.item())
                if result.names[cls] == 'boat' and track_id not in seen:
                    seen.add(track_id)
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    crop_img = frame[y1:y2, x1:x2]
                    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
                    filename = f"poc_boat_{track_id}_{ts}.jpg"
                    crop_path = os.path.abspath(os.path.join(CROPS_DIR, filename))
                    cv2.imwrite(crop_path, crop_img)
                    with sqlite3.connect(DB_PATH) as conn:
                        conn.execute("INSERT INTO detections (track_id, timestamp, image_path) VALUES (?, ?, ?)",
                                     (track_id, ts, filename))
                    print(f"[!] NAVIO IDENTIFICADO! RG: {track_id}. Foto anotada: {filename}")
                    count += 1
    cv2.imshow('Monitoramento Porto de Santos - PoC', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()
print("\nEtapa de detecção concluída com sucesso.")

### 2. Processamento VLM (Vision Language Model)
Vamos percorrer o banco de dados e analisar cada imagem pendente localmente.

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    df_pendentes = pd.read_sql_query("SELECT * FROM detections WHERE status = 'pendente'", conn)
    for idx, row in df_pendentes.iterrows():
        full_image_path = os.path.join(CROPS_DIR, row['image_path'])
        print(f"[ANALISANDO] Foto {row['image_path']} localmente com Ollama ({VLM_MODEL})...")
        descricao = analyze_image_with_vlm(full_image_path)
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute("UPDATE detections SET vlm_description = ?, status = 'concluido' WHERE id = ?",
                (descricao, row['id']))
        print(f"[IA RESPONDEU]: {descricao}\n")
print("O especialista terminou de analisar todas as fotos!")

### 3. Sumarização do Dia

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    df_final = pd.read_sql_query("SELECT vlm_description FROM detections WHERE status = 'concluido'", conn)

todas_descricoes = "\n".join(df_final['vlm_description'].tolist())
prompt_summary = f"""
Abaixo estão as notas técnicas de todos os navios que passaram pelo Porto de Santos hoje:
{todas_descricoes}

Com base nessas informações, escreva um parágrafo de resumo executivo.
O tom deve ser profissional. Foque em resumir qual foi o principal tipo de movimentação detectada hoje.
"""

try:
    response_llm = ollama.chat(
        model=LLM_MODEL,
        messages=[{'role': 'user', 'content': prompt_summary}]
    )
    resumo_texto = response_llm['message']['content']
except Exception as e:
    resumo_texto = f"Erro na sumarização local: {e}"

print("\n" + "="*50)
print("       RELATÓRIO DE INTELIGÊNCIA MARÍTIMA (LOCAL)")
print("="*50)
print(resumo_texto)
print("="*50)